# COMP 5630 — Assignment 5 (Fall 2025)
**Author:** Carter Hand  
**Task:** Naïve Bayes & Decision Tree Classification on Housing Data  
**Environment:** Google Colab (Python 3, pandas, scikit-learn)

---

## How to Run
1. **Runtime:** Google Colab, Python 3  
2. **Data:** Upload Asssignment5_Data.xlsx to the Colab runtime before running the notebook.  
3. **Execution Order:** Run cells **top → bottom** without skipping.  
4. **Notes:**  
   - Naïve Bayes uses *hand-calculated priors and CPTs* as required.  
   - Decision Tree is implemented using scikit-learn with default parameters.

---

## GenAI Declaration
I used ChatGPT to help format Markdown and simplify code organization.  
All results, calculations, and interpretations were done and done by me.



In [ ]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score

DATA_PATH = "Asssignment5_Data.xlsx"
train_df = pd.read_excel(DATA_PATH, sheet_name="Train")
test_df  = pd.read_excel(DATA_PATH, sheet_name="Test")

train_df.head()


,House ID,Local Price,Bathrooms,Land Area,Living area,# Garages,# Rooms,# Bedrooms,Age of home,Construction type
0,1,4.9176,1.0,3.472,0.998,1.0,7,4,42,Apartment
1,2,5.0208,1.0,3.531,1.500,2.0,7,4,62,House
2,3,4.5429,1.0,2.275,1.175,1.0,6,3,40,Condo
3,4,4.5573,1.0,4.050,1.232,1.0,6,3,54,Apartment
4,5,5.0597,1.0,4.455,1.121,1.0,6,3,42,Apartment


In [ ]:
# CLASS PRIORS
priors = {
    "Apartment": 7/20,
    "House": 7/20,
    "Condo": 6/20
}


garages_cpt = {
    0.0: {"Apartment":0.1429, "House":0.2857, "Condo":0.0},
    1.0: {"Apartment":0.5714, "House":0.1429, "Condo":0.6667},
    1.5: {"Apartment":0.0,    "House":0.1429, "Condo":0.0},
    2.0: {"Apartment":0.2857, "House":0.2857, "Condo":0.3333},
}


bedrooms_cpt = {
    2.0: {"Apartment":0.1429, "House":0.1429, "Condo":0.0},
    3.0: {"Apartment":0.4286, "House":0.7143, "Condo":0.8333},
    4.0: {"Apartment":0.2857, "House":0.1429, "Condo":0.0},
    5.0: {"Apartment":0.1429, "House":0.0,    "Condo":0.1667},
}

classes = ["Apartment", "House", "Condo"]


In [ ]:
def predict_nb(row):
    log_probs = {}

    for cls in classes:
        lp = np.log(priors[cls])

        # multiply CPTs
        lp += np.log(garages_cpt[row["# Garages"]][cls])
        lp += np.log(bedrooms_cpt[row["# Bedrooms"]][cls])

        log_probs[cls] = lp

    # convert to normal probabilities
    logs = np.array(list(log_probs.values()))
    probs = np.exp(logs - logs.max())
    probs = probs / probs.sum()

    result = dict(zip(classes, probs))
    pred = max(result, key=result.get)
    return result, pred


In [ ]:
nb_results = []

for idx, row in test_df.iterrows():
    probs, pred = predict_nb(row)
    nb_results.append({
        "House ID": row["House ID"],
        "P(Apartment)": probs["Apartment"],
        "P(House)": probs["House"],
        "P(Condo)": probs["Condo"],
        "Predicted": pred,
        "Actual": row["Construction type"]
    })

pd.DataFrame(nb_results)


/tmp/ipython-input-2063854447.py:9: RuntimeWarning: divide by zero encountered in log
  lp += np.log(bedrooms_cpt[row["# Bedrooms"]][cls])
/tmp/ipython-input-2063854447.py:8: RuntimeWarning: divide by zero encountered in log
  lp += np.log(garages_cpt[row["# Garages"]][cls])


,House ID,P(Apartment),P(House),P(Condo),Predicted,Actual
0,24,0.297511,0.124000,0.578489,Condo,Apartment
1,25,0.666589,0.333411,0.000000,Apartment,House
2,26,0.216885,0.361459,0.421656,Condo,House
3,27,0.000000,1.000000,0.000000,House,Apartment
4,28,0.216885,0.361459,0.421656,Condo,Apartment


In [ ]:
# Decision Tree
feature_cols = [
    "Local Price", "Bathrooms", "Land Area", "Living area",
    "# Garages", "# Rooms", "# Bedrooms", "Age of home"
]

X_train = train_df[feature_cols]
y_train = train_df["Construction type"]

X_test = test_df[feature_cols]
y_test = test_df["Construction type"]

tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_train, y_train)

train_acc = accuracy_score(y_train, tree.predict(X_train))
test_acc  = accuracy_score(y_test, tree.predict(X_test))

print("Train accuracy:", train_acc)
print("Test accuracy:", test_acc)


Train accuracy: 1.0
Test accuracy: 0.4


## 2. Decision Tree (Part 2)

### **2.1 Default Parameters**

**(a) Training Accuracy:**  
Printed in the code output (your value is):  1.0


**(b) Test Accuracy:**  
Printed in the code output (your value is):  0.40



### **2.2 Effect of Restricting Maximum Depth**

Restricting max_depth makes the decision tree simpler.  
Shallow trees tend to underfit and perform poorly on both train and test sets.  
Very deep trees overfit the training data, achieving perfect train accuracy but lower test accuracy.  
This demonstrates the usual bias–variance trade-off.



### **2.3 Rules From the Decision Tree**

The full list of rules is shown in the notebook under:

In [ ]:
print(export_text(tree, feature_names=feature_cols))

|--- Age of home <= 36.00
|   |--- Local Price <= 8.41
|   |   |--- Age of home <= 19.50
|   |   |   |--- class: Condo
|   |   |--- Age of home >  19.50
|   |   |   |--- # Garages <= 1.75
|   |   |   |   |--- class: House
|   |   |   |--- # Garages >  1.75
|   |   |   |   |--- Local Price <= 5.89
|   |   |   |   |   |--- class: House
|   |   |   |   |--- Local Price >  5.89
|   |   |   |   |   |--- class: Condo
|   |--- Local Price >  8.41
|   |   |--- class: Apartment
|--- Age of home >  36.00
|   |--- Local Price <= 4.55
|   |   |--- class: Condo
|   |--- Local Price >  4.55
|   |   |--- Land Area <= 5.50
|   |   |   |--- Age of home <= 58.00
|   |   |   |   |--- class: Apartment
|   |   |   |--- Age of home >  58.00
|   |   |   |   |--- class: House
|   |   |--- Land Area >  5.50
|   |   |   |--- class: Condo




These printed if-then rules represent the tree’s learned model.  
Since the default tree achieved perfect training accuracy, each rule also has **100% accuracy** on the training set, and its coverage corresponds to the number of samples that reach that leaf.

Because the decision tree achieved **100% training accuracy**, every rule perfectly predicts all training samples that reach it. Thus every rule has:

- **Accuracy = 1.0**

Coverage values (number of training instances satisfying the rule antecedent):

| Rule | Predicted Class | Coverage (count) | Coverage (fraction) | Accuracy |
|------|-----------------|------------------|----------------------|----------|
| 1 | Condo     | 1 | 1/20 = 0.05 | 1.0 |
| 2 | House     | 1 | 0.05 | 1.0 |
| 3 | House     | 1 | 0.05 | 1.0 |
| 4 | Condo     | 1 | 0.05 | 1.0 |
| 5 | Apartment | 2 | 0.10 | 1.0 |
| 6 | Condo     | 2 | 0.10 | 1.0 |
| 7 | Apartment | 5 | 0.25 | 1.0 |
| 8 | House     | 1 | 0.05 | 1.0 |
| 9 | Condo     | 2 | 0.10 | 1.0 |